In [5]:
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage

# Tool Creation

In [7]:
@tool
def multiply(a:int,b:int) -> int:
    '''This function takes 2 integers as input and returns the products.'''
    return a * b

# Tool Binding

-> This step binds the tool with llm.

-> Whenever LLM thinks that a particular user query may require the need of tool, then LLM will call the tools
   form the given set of binded tools.

In [8]:
llm = ChatOpenAI()

In [10]:
llm_with_tools = llm.bind_tools(tools = [multiply])

In [11]:
llm_with_tools

_ChatModelBinding(bound=ChatOpenAI(metadata={'lc_versions': {'langchain-core': '1.4.9', 'langchain': '1.3.12', 'langchain-openai': '1.3.5'}}, output_version=None, profile={'name': 'GPT-3.5-turbo', 'release_date': '2023-03-01', 'last_updated': '2023-11-06', 'open_weights': False, 'max_input_tokens': 16385, 'max_output_tokens': 4096, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': False, 'structured_output': False, 'attachment': False, 'temperature': True, 'image_url_inputs': False, 'pdf_inputs': False, 'pdf_tool_message': False, 'image_tool_message': False, 'tool_choice': True, 'tool_call_streaming': True}, client=<openai.resources.chat.completions.completions.Completions object at 0x0000020C9F5EC830>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x0000020C9F5ED2B0>, ro

# Tool Calling

Tool calling allows an LLM to interact with external functions, APIs, databases, or other systems.

The LLM does not execute the tool.

The LLM decides:

Whether a tool is needed.

Which tool to use.

What arguments to provide.

The LLM generates a structured tool call, for example:

{
  "name": "multiply",
  "arguments": {
    "a": 10,
    "b": 5
  }
}

The application/tool runtime receives this request and actually executes the function.

The tool result is then sent back to the LLM.

The LLM uses the result to generate the final response or decide whether another tool is needed.

In [13]:
llm_with_tools.invoke('Hi how can you help me ?')

AIMessage(content="Hello! I can help you with a variety of tasks. Let me know what you need assistance with, and I'll do my best to help you out.", additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 33, 'prompt_tokens': 61, 'total_tokens': 94, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'id': 'chatcmpl-EAsFRyiIqW4tstvyTp25MqbZKSgdQ', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019fe56c-6d6b-7983-8cca-76642cc243dd-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 61, 'output_tokens': 33, 'total_tokens': 94, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning

In [19]:
tool_call = llm_with_tools.invoke('Can you multiply 8 by 3').tool_calls[0]

In [20]:
tool_call

{'name': 'multiply',
 'args': {'a': 8, 'b': 3},
 'id': 'call_ibrscLu4qyymMBX4LREn68JX',
 'type': 'tool_call'}

# Tool Execution

-> It is the programmers responsibility to execute the tool after LLM Suggests to use the tool.

In [22]:
multiply.invoke(tool_call)

ToolMessage(content='24', name='multiply', tool_call_id='call_ibrscLu4qyymMBX4LREn68JX')

Now it is the applications responsibility to send the response back to the llm to show the result but for that LLM
needs to store the contex.

# Entire Tool Usage Flow with Context

In [73]:
query = HumanMessage(content = "Hi How can you help me ?")

In [74]:
messages = [query]

In [75]:
response = llm_with_tools.invoke(messages)

In [76]:
messages.append(response)

In [77]:
messages

[HumanMessage(content='Hi How can you help me ?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='Hello! I can help you with various tasks. What do you need assistance with today?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 19, 'prompt_tokens': 61, 'total_tokens': 80, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'id': 'chatcmpl-EAsZ0k9Uy6RsdNmgATlR5g97ilnn5', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019fe57e-f37e-7702-876e-d119992c0a19-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 61, 'output_tokens': 19, 'total_tokens': 80, 'input_token_details': {'audio': 0, 'cache_read': 

In [78]:
calc_query = HumanMessage(content = "Can you multiply 45 by 2")

In [79]:
messages.append(calc_query)

In [80]:
messages

[HumanMessage(content='Hi How can you help me ?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='Hello! I can help you with various tasks. What do you need assistance with today?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 19, 'prompt_tokens': 61, 'total_tokens': 80, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'id': 'chatcmpl-EAsZ0k9Uy6RsdNmgATlR5g97ilnn5', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019fe57e-f37e-7702-876e-d119992c0a19-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 61, 'output_tokens': 19, 'total_tokens': 80, 'input_token_details': {'audio': 0, 'cache_read': 

In [81]:
response = llm_with_tools.invoke(messages)

In [82]:
messages.append(response)

In [83]:
messages

[HumanMessage(content='Hi How can you help me ?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='Hello! I can help you with various tasks. What do you need assistance with today?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 19, 'prompt_tokens': 61, 'total_tokens': 80, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'id': 'chatcmpl-EAsZ0k9Uy6RsdNmgATlR5g97ilnn5', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019fe57e-f37e-7702-876e-d119992c0a19-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 61, 'output_tokens': 19, 'total_tokens': 80, 'input_token_details': {'audio': 0, 'cache_read': 

In [84]:
response.tool_calls[0]

{'name': 'multiply',
 'args': {'a': 45, 'b': 2},
 'id': 'call_AXoarTpAxMDZ6oUSyRiAcvoN',
 'type': 'tool_call'}

In [85]:
tool_call = response.tool_calls[0]

In [86]:
multiply.invoke(tool_call)

ToolMessage(content='90', name='multiply', tool_call_id='call_AXoarTpAxMDZ6oUSyRiAcvoN')

In [87]:
tool_response = multiply.invoke(tool_call)

In [88]:
messages.append(tool_response)

In [89]:
llm_with_tools.invoke(messages)

AIMessage(content='The result of multiplying 45 by 2 is 90. If you need further assistance or have any other questions, feel free to let me know!', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 32, 'prompt_tokens': 120, 'total_tokens': 152, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'id': 'chatcmpl-EAsZQW3Xx4mAQb9F9Vqnm8pvbnYi1', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019fe57f-5cc6-7080-a614-db2b15ea90db-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 120, 'output_tokens': 32, 'total_tokens': 152, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reaso